In [58]:
import pandas as pd

In [41]:
contacts = pd.read_csv( r"C:\Program Files\Projects\Fundraising Data set\kaggle_dataset\contacts.csv"
)

contacts.head()

,Contact_Report_ID,Donor_ID,Contact_Date,Contact_Type,Author,Report_Text,Outcome_Category
0,500000,770487,2015-02-05,Event,Lisa Thompson,Attempted to contact Kelly Rodriguez regarding...,Negative
1,500001,388389,2019-01-23,Meeting,James Wilson,Multiple outreach attempts to Andrew Jones abo...,Negative
2,500002,872246,2025-04-28,Event,Maria Gonzalez,Spoke with Diane Adams at President's Circle E...,Positive
3,500003,207473,2025-03-31,Event,Maria Gonzalez,Met with Deborah Hill at their home to discuss...,Negative
4,500004,809570,2023-02-15,Event,David Park,Solicitation meeting with Paul Smith for schol...,Positive


In [42]:
contacts['rag_text'] = (
    "Donor ID: " + contacts['Donor_ID'].astype(str) +
    "\nContact Date: " + contacts['Contact_Date'].astype(str) +
    "\nContact Type: " + contacts['Contact_Type'].astype(str) +
    "\nOutcome: " + contacts['Outcome_Category'].astype(str) +
    "\nReport: " + contacts['Report_Text'].astype(str)
)

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=20000
)

tfidf_matrix = vectorizer.fit_transform(contacts['rag_text'])

In [44]:
def retrieve_contact_reports(query, top_k=5):

    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vec,
        tfidf_matrix
    ).flatten()

    if similarities.max() < 0.05:
        return "No relevant documents found."

    top_indices = similarities.argsort()[-top_k:][::-1]

    return contacts.iloc[top_indices][
        [
            'Donor_ID',
            'Contact_Date',
            'Contact_Type',
            'Outcome_Category',
            'Report_Text'
        ]
    ]

In [45]:
retrieve_contact_reports(
    "positive meeting with major gift potential",
    top_k=5
)

,Donor_ID,Contact_Date,Contact_Type,Outcome_Category,Report_Text
25688,162782,2022-08-29,Meeting,Positive,Formal solicitation meeting with Thomas Smith ...
267714,144432,2024-01-07,Meeting,Positive,Formal solicitation meeting with Amy Johnson f...
306785,312408,2022-06-30,Meeting,Positive,Formal solicitation meeting with Janet Smith f...
231444,739036,2024-10-20,Meeting,Positive,Formal solicitation meeting with Janet Martine...
324338,384826,2025-01-19,Meeting,Positive,Formal solicitation meeting with Laura Smith f...


In [46]:
def retrieve_contact_reports(query, top_k=5, min_score=0.05):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    if similarities.max() < min_score:
        return "No relevant contact reports found."

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = contacts.iloc[top_indices][
        ['Donor_ID', 'Contact_Date', 'Contact_Type', 'Outcome_Category', 'Report_Text']
    ].copy()

    results['similarity_score'] = similarities[top_indices]

    return results

In [47]:
retrieve_contact_reports("scholarships alumni events")

,Donor_ID,Contact_Date,Contact_Type,Outcome_Category,Report_Text,similarity_score
243200,239818,2024-10-12,Email,Negative,Spoke with Thomas Smith at Alumni Reception. D...,0.254959
165504,620374,2024-09-04,Email,Negative,Connected with Thomas Johnson during Alumni Re...,0.253253
42122,571143,2023-12-09,Phone Call,Positive,Spoke with Janet Smith at Alumni Reception. Di...,0.253090
147946,947078,2025-06-04,Phone Call,Positive,Spoke with Sarah Smith at Alumni Reception. Di...,0.252039
132819,625901,2025-07-09,Meeting,Negative,Connected with Thomas Johnson during Alumni Re...,0.251918


In [48]:
def create_donor_briefing(query, top_k=5):
    results = retrieve_contact_reports(query, top_k=top_k)

    if isinstance(results, str):
        return results

    briefing = f"Donor Briefing for Query: {query}\n\n"

    for _, row in results.iterrows():
        briefing += f"""
Donor ID: {row['Donor_ID']}
Contact Date: {row['Contact_Date']}
Contact Type: {row['Contact_Type']}
Outcome: {row['Outcome_Category']}
Relevance Score: {row['similarity_score']:.3f}

Summary Source:
{row['Report_Text']}

---
"""

    return briefing

In [49]:
print(create_donor_briefing("scholarships alumni events"))

Donor Briefing for Query: scholarships alumni events


Donor ID: 239818
Contact Date: 2024-10-12
Contact Type: Email
Outcome: Negative
Relevance Score: 0.255

Summary Source:
Spoke with Thomas Smith at Alumni Reception. Discussed capital campaign. Thomas Smith will discuss with spouse.

---

Donor ID: 620374
Contact Date: 2024-09-04
Contact Type: Email
Outcome: Negative
Relevance Score: 0.253

Summary Source:
Connected with Thomas Johnson during Alumni Reception. Brief conversation about annual giving. Thomas Johnson requested proposal.

---

Donor ID: 571143
Contact Date: 2023-12-09
Contact Type: Phone Call
Outcome: Positive
Relevance Score: 0.253

Summary Source:
Spoke with Janet Smith at Alumni Reception. Discussed scholarship support. Janet Smith requested proposal.

---

Donor ID: 947078
Contact Date: 2025-06-04
Contact Type: Phone Call
Outcome: Positive
Relevance Score: 0.252

Summary Source:
Spoke with Sarah Smith at Alumni Reception. Discussed planned giving. Sarah Smith indica

In [50]:
def prospect_search_summary(query, top_k=10):
    results = prospect_search(query, top_k=top_k)

    if isinstance(results, str):
        return results

    summary = f"Prospect Search: {query}\n\n"
    summary += f"Found {len(results)} relevant contact reports.\n\n"

    for _, row in results.iterrows():
        summary += (
            f"Donor {row['Donor_ID']} | "
            f"{row['Contact_Date']} | "
            f"{row['Outcome_Category']} | "
            f"Score: {row['similarity_score']:.3f}\n"
            f"Note: {row['Report_Text'][:300]}...\n\n"
        )

    return summary

In [51]:
def build_rag_context(query, top_k=5, min_score=0.05):
    results = retrieve_contact_reports(
        query=query,
        top_k=top_k,
        min_score=min_score
    )

    if isinstance(results, str):
        return results

    context = "RELEVANT CONTACT REPORTS:\n\n"

    for i, row in results.iterrows():
        context += f"""
Report {i+1}
Donor ID: {row['Donor_ID']}
Date: {row['Contact_Date']}
Contact Type: {row['Contact_Type']}
Outcome: {row['Outcome_Category']}
Relevance Score: {row['similarity_score']:.3f}

Report Text:
{row['Report_Text']}

---
"""

    return context

In [52]:
def build_rag_prompt(query, top_k=5):
    context = build_rag_context(query, top_k=top_k)

    if context == "No relevant contact reports found.":
        return context

    prompt = f"""
You are an advancement analytics assistant.

Use the contact reports below to answer the user's question.

Rules:
- Only use the provided contact reports.
- Do not invent facts.
- Return donor IDs when relevant.
- Be concise.
- If the evidence is weak, say so.

User Question:
{query}

{context}

Answer:
"""

    return prompt

In [53]:
prompt = build_rag_prompt(
    "Find prospects interested in scholarships",
    top_k=5
)

print(prompt)


You are an advancement analytics assistant.

Use the contact reports below to answer the user's question.

Rules:
- Only use the provided contact reports.
- Do not invent facts.
- Return donor IDs when relevant.
- Be concise.
- If the evidence is weak, say so.

User Question:
Find prospects interested in scholarships

RELEVANT CONTACT REPORTS:


Report 180910
Donor ID: 478172
Date: 2024-12-07
Contact Type: Meeting
Outcome: Positive
Relevance Score: 0.219

Report Text:
Phone conversation with Thomas Smith about potential support for annual giving. Thomas Smith not interested in this initiative.

---

Report 8881
Donor ID: 170601
Date: 2023-09-02
Contact Type: Event
Outcome: Positive
Relevance Score: 0.215

Report Text:
Met with Thomas Smith to discuss building project. Unfortunately, Thomas Smith not interested in this initiative.

---

Report 319573
Donor ID: 208483
Date: 2020-02-07
Contact Type: Meeting
Outcome: Positive
Relevance Score: 0.215

Report Text:
Met with Thomas Johnson to

In [54]:
def rag_assistant_prompt(query, top_k=5):
    prompt = build_rag_prompt(query, top_k=top_k)
    print(prompt)

In [55]:
rag_assistant_prompt("donors who want to be removed from outreach", top_k=5)


You are an advancement analytics assistant.

Use the contact reports below to answer the user's question.

Rules:
- Only use the provided contact reports.
- Do not invent facts.
- Return donor IDs when relevant.
- Be concise.
- If the evidence is weak, say so.

User Question:
donors who want to be removed from outreach

RELEVANT CONTACT REPORTS:


Report 35123
Donor ID: 794235
Date: 2025-07-06
Contact Type: Event
Outcome: Negative
Relevance Score: 0.351

Report Text:
Multiple outreach attempts to Scott Smith about research funding. Scott Smith did not respond to email outreach. Considering pause in contact.

---

Report 101960
Donor ID: 638874
Date: 2024-12-13
Contact Type: Meeting
Outcome: Negative
Relevance Score: 0.350

Report Text:
Multiple outreach attempts to Sarah Smith about athletics. Sarah Smith did not respond to email outreach. Considering pause in contact.

---

Report 134980
Donor ID: 297449
Date: 2023-05-20
Contact Type: Meeting
Outcome: Positive
Relevance Score: 0.348


In [59]:
def prospect_search(query, top_k=20, min_score=0.05):
    results = retrieve_contact_reports(
        query=query,
        top_k=top_k,
        min_score=min_score
    )

    if isinstance(results, str):
        return results

    return results[
        [
            'Donor_ID',
            'Contact_Date',
            'Contact_Type',
            'Outcome_Category',
            'similarity_score',
            'Report_Text'
        ]
    ].sort_values(
        'similarity_score',
        ascending=False
    )

In [61]:
def rag_style_summary(query, top_k=5):
    results = prospect_search(query, top_k=top_k)

    if isinstance(results, str):
        return results

    print(f"Query: {query}")
    print(f"Found {len(results)} relevant contact reports.\n")

    for _, row in results.iterrows():
        print(f"Donor ID: {row['Donor_ID']}")
        print(f"Date: {row['Contact_Date']}")
        print(f"Outcome: {row['Outcome_Category']}")
        print(f"Relevance: {row['similarity_score']:.3f}")
        print(f"Why matched: {row['Report_Text'][:250]}...")
        print("-" * 60)

In [62]:
rag_style_summary("interested in scholarships", top_k=5)

Query: interested in scholarships
Found 5 relevant contact reports.

Donor ID: 478172
Date: 2024-12-07
Outcome: Positive
Relevance: 0.219
Why matched: Phone conversation with Thomas Smith about potential support for annual giving. Thomas Smith not interested in this initiative....
------------------------------------------------------------
Donor ID: 170601
Date: 2023-09-02
Outcome: Positive
Relevance: 0.215
Why matched: Met with Thomas Smith to discuss building project. Unfortunately, Thomas Smith not interested in this initiative....
------------------------------------------------------------
Donor ID: 208483
Date: 2020-02-07
Outcome: Positive
Relevance: 0.215
Why matched: Met with Thomas Johnson to discuss athletics. Unfortunately, Thomas Johnson not interested in this initiative....
------------------------------------------------------------
Donor ID: 743382
Date: 2025-08-02
Outcome: Positive
Relevance: 0.214
Why matched: Phone conversation with Thomas Smith about potential suppo

In [ ]:
rag_style_summary("interested in scholarships", top_k=5)

Query: interested in scholarships
Found 5 relevant contact reports.

Donor ID: 478172
Date: 2024-12-07
Outcome: Positive
Relevance: 0.219
Why matched: Phone conversation with Thomas Smith about potential support for annual giving. Thomas Smith not interested in this initiative....
------------------------------------------------------------
Donor ID: 170601
Date: 2023-09-02
Outcome: Positive
Relevance: 0.215
Why matched: Met with Thomas Smith to discuss building project. Unfortunately, Thomas Smith not interested in this initiative....
------------------------------------------------------------
Donor ID: 208483
Date: 2020-02-07
Outcome: Positive
Relevance: 0.215
Why matched: Met with Thomas Johnson to discuss athletics. Unfortunately, Thomas Johnson not interested in this initiative....
------------------------------------------------------------
Donor ID: 743382
Date: 2025-08-02
Outcome: Positive
Relevance: 0.214
Why matched: Phone conversation with Thomas Smith about potential suppo

In [64]:
rag_style_summary("donors who want to be removed from outreach", top_k=5)

Query: donors who want to be removed from outreach
Found 5 relevant contact reports.

Donor ID: 794235
Date: 2025-07-06
Outcome: Negative
Relevance: 0.351
Why matched: Multiple outreach attempts to Scott Smith about research funding. Scott Smith did not respond to email outreach. Considering pause in contact....
------------------------------------------------------------
Donor ID: 638874
Date: 2024-12-13
Outcome: Negative
Relevance: 0.350
Why matched: Multiple outreach attempts to Sarah Smith about athletics. Sarah Smith did not respond to email outreach. Considering pause in contact....
------------------------------------------------------------
Donor ID: 297449
Date: 2023-05-20
Outcome: Positive
Relevance: 0.348
Why matched: Multiple outreach attempts to Janet Smith about scholarship support. Janet Smith did not respond to email outreach. Considering pause in contact....
------------------------------------------------------------
Donor ID: 753456
Date: 2025-10-19
Outcome: Negative

In [65]:
rag_style_summary("positive meeting with major gift potential", top_k=5)

Query: positive meeting with major gift potential
Found 5 relevant contact reports.

Donor ID: 162782
Date: 2022-08-29
Outcome: Positive
Relevance: 0.344
Why matched: Formal solicitation meeting with Thomas Smith for $12 gift to building project. Thomas Smith committed to gift....
------------------------------------------------------------
Donor ID: 144432
Date: 2024-01-07
Outcome: Positive
Relevance: 0.335
Why matched: Formal solicitation meeting with Amy Johnson for $1 gift to athletics. Amy Johnson committed to gift....
------------------------------------------------------------
Donor ID: 312408
Date: 2022-06-30
Outcome: Positive
Relevance: 0.331
Why matched: Formal solicitation meeting with Janet Smith for $14 gift to research funding. Janet Smith committed to gift....
------------------------------------------------------------
Donor ID: 739036
Date: 2024-10-20
Outcome: Positive
Relevance: 0.331
Why matched: Formal solicitation meeting with Janet Martinez for $8 gift to scholars